In [1]:
!pip install -q SoccerNet roboflow ultralytics dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7

In [2]:
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="SoccerNet")
mySoccerNetDownloader.downloadDataTask(task="tracking", split=["train"])
#mySoccerNetDownloader.downloadDataTask(task="tracking-2023", split=["train", "test", "challenge"])

Stitching 750 frames into test_video_061.mp4...
Video compilation complete!


In [3]:
import cv2
import glob

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

import zipfile
import pandas as pd
import cv2
import shutil
import numpy as np
import os
import configparser


def extract_zip(zip_path, extract_to):
    """
    Extracts a ZIP file to the specified directory.

    :param zip_path: Path to the .zip file
    :param extract_to: Directory where files will be extracted
    """
    try:
        # Validate that the file exists
        if not os.path.isfile(zip_path):
            print(f"Error: File '{zip_path}' does not exist.")
            return

        # Ensure the extraction directory exists
        os.makedirs(extract_to, exist_ok=True)

        # Open and extract the ZIP file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            print(f"Successfully extracted '{zip_path}' to '{extract_to}'")

    except zipfile.BadZipFile:
        print(f"Error: '{zip_path}' is not a valid ZIP file.")
    except PermissionError:
        print("Error: Permission denied while accessing files.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def frames_to_video(images_dir, output_video_path, fps=25):
    # Grab all jpeg images and sort them numerically so the video plays in order
    image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))

    if not image_files:
        print(f"No images found in {images_dir}")
        return

    # Read the first image to dynamically extract widescreen dimensions
    first_frame = cv2.imread(image_files[0])
    height, width, layers = first_frame.shape

    # Initialize the video writer with the MP4V codec
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"Stitching {len(image_files)} frames into {output_video_path}...")
    for image_file in image_files:
        frame = cv2.imread(image_file)
        video.write(frame)

    video.release()
    print("Video compilation complete!")

def extract_player_crops(yolo_results, frame, player_class_id=1):

    player_data = []

    if yolo_results is None or yolo_results.boxes is None:
        return player_data
    boxes = yolo_results.boxes
    frame_h, frame_w = frame.shape[:2]
    for i in range(len(boxes)):
        if boxes.id is None:
            continue
        cls_id = int(boxes.cls[i].item())
        if cls_id != player_class_id:
            continue

        track_id = int(boxes.id[i].item())
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().astype(int)

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(frame_w, x2)
        y2 = min(frame_h, y2)

        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            player_data.append({
                'id': track_id,
                'bbox': [x1, y1, x2, y2],
                'crop': crop
            })

    return player_data
def assign_teams_by_anchor(players_crop, resize_dim=(64, 64), anchors=None):
    if not players_crop:
        return [], anchors

    feature_list = []
    valid_players = []
    for player in players_crop:
        crop = player['crop']
        if crop is None or crop.size == 0:
            continue
        height, width, _ = crop.shape
        torso_crop = crop[0:int(height * 0.6), :]

        if torso_crop.size == 0:
            continue

        resized = cv2.resize(torso_crop, resize_dim)
        hsv_resized = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)
        flattened_features = hsv_resized.flatten()

        feature_list.append(flattened_features)
        valid_players.append(player)

    if not feature_list:
        return players_crop, anchors

    feature_matrix = np.array(feature_list)

    if anchors is None:
        print("Automatically initializing team color anchors from the first frame...")
        kmeans = KMeans(n_clusters=2, init='k-means++', random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(feature_matrix)

        anchors = kmeans.cluster_centers_

        for idx, player in enumerate(valid_players):
            player['team_cluster'] = int(cluster_labels[idx])

    else:
        for player_idx, feature_vector in enumerate(feature_matrix):
            dist_to_team_0 = np.linalg.norm(feature_vector - anchors[0])
            dist_to_team_1 = np.linalg.norm(feature_vector - anchors[1])
            assigned_team = 0 if dist_to_team_0 < dist_to_team_1 else 1
            valid_players[player_idx]['team_cluster'] = assigned_team

    return valid_players, anchors

def extract_jersey_number(results):
    boxes = results.boxes
    if boxes is None or len(boxes) == 0:
        return None
    detections = []
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        #conf = float(box.conf[0].item())
        cls_id = int(box.cls[0].item())
        x_center = (x1 + x2) / 2.0

        detections.append({
            'class': cls_id,
            'x_center': x_center,
        })
        #print(detections)


    # Sort explicitly by the horizontal center coordinate
    sorted_left_to_right = sorted(detections, key=lambda d: d['x_center'])
    digit = ""
    for det in sorted_left_to_right:
        digit+= str(det['class'])
    return int(digit)

def extract_jersey_number(results):
    boxes = results.boxes
    if boxes is None or len(boxes) == 0:
        return None, 0
    detections = []
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].item())
        cls_id = int(box.cls[0].item())
        x_center = (x1 + x2) / 2.0

        detections.append({
            'class': cls_id,
            'conf': conf,
            'x_center': x_center,
            #'box': (x1, y1, x2, y2)
        })
    #print(detections)


    # Sort explicitly by the horizontal center coordinate
    sorted_left_to_right = sorted(detections, key=lambda d: d['x_center'])
    digit = ""
    avg_conf = 0
    for det in sorted_left_to_right:
        digit+= str(det['class'])
        avg_conf += (det["conf"]/len(sorted_left_to_right))

    return int(digit), avg_conf

In [4]:
zip_file_path = "/content/SoccerNet/tracking/train.zip"
output_directory = "train"

extract_zip(zip_file_path, output_directory)



# Usage: Point to one of your downloaded sequence image directories
images_input = "/content/train/train/SNMOT-061/img1"
video_output = "test_video_061.mp4"

frames_to_video(images_input, video_output, fps=25)


Successfully extracted '/content/SoccerNet/tracking/train.zip' to 'train'
Stitching 750 frames into test_video_061.mp4...
Video compilation complete!


In [5]:
frames_to_video("/content/train/train/SNMOT-066/img1", "SNMOT-066.mp4", fps=25)

Stitching 750 frames into SNMOT-066.mp4...
Video compilation complete!


In [6]:
#import gdown


#gdown.download(f'https://drive.google.com/uc?id=1slQS_CdGYMbFiH7xBFsNxrtvBFyy7knG', "best-large-image.pt", quiet=False)
#gdown.download(f'https://drive.google.com/uc?id=1bOOfGYGekji_r37Z8FBcJoYPvD67aspK', "test_video_116.mp4", quiet=False)
#gdown.download(f'https://drive.google.com/uc?id=1pENrq7YcfYacmB0iprCkSeo4DAf2FadC', "best.pt", quiet=False)

In [7]:
from ultralytics import YOLO
model = YOLO("https://github.com/AgabaEmbedded/Soccer-Tracking/releases/download/1.0/best-large-image.pt")
jn_model = YOLO("https://github.com/AgabaEmbedded/Soccer-Tracking/releases/download/1.1/best_large.pt")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [24]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from ultralytics import YOLO
import json
from collections import Counter

# ==========================================
# CONFIGURATION & HYPERPARAMETERS
# ==========================================
VIDEO_PATH = "/content/SNMOT-066.mp4"
METADATA_SAVE_PATH = "/content/video_tracking_metadata.json"
OUTPUT_VIDEO_PATH = "/content/output_tracked_match2.mp4"

# Thresholds
PLAYER_CONF_THRESH = 0.25
DIGIT_CONF_THRESH = 0.40
SINGLE_FRAME_DIGIT_CONF = 0.50  # Min avg confidence to accept frame digit detection
MIN_GLOBAL_ACCUMULATED_WEIGHT = 0.5  # Pass 2: Min sum weight to confirm a jersey

CLASS_COLORS = {0: (0, 0, 255), 1: (255, 255, 0), 3: (0, 255, 255)}
TEAM_COLORS = {0: (255, 0, 0), 1: (0, 255, 0)}


# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_jersey_number_from_yolo(jn_result, conf_thresh=DIGIT_CONF_THRESH):
    """
    Parses YOLOv11 digit detections from a torso crop, sorts them horizontally (left-to-right),
    and reconstructs multi-digit numbers.
    """
    boxes = jn_result.boxes
    if boxes is None or len(boxes) == 0:
        return None, 0.0

    detections = []
    for i in range(len(boxes)):
        conf = float(boxes.conf[i].item())
        if conf < conf_thresh:
            continue

        cls_id = int(boxes.cls[i].item())
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
        x_center = (x1 + x2) / 2.0

        detections.append({
            'digit': str(cls_id),
            'x_center': x_center,
            'conf': conf
        })

    if not detections:
        return None, 0.0

    # Sort left-to-right spatially
    detections.sort(key=lambda d: d['x_center'])

    digit_string = "".join([d['digit'] for d in detections])
    avg_confidence = sum([d['conf'] for d in detections]) / len(detections)

    return digit_string, avg_confidence

def resolve_team_by_majority(metadata):
    team_counters = {}

    for frame_idx, objs in metadata["frames"].items():
        for obj in objs:
            track_id = obj.get("track_id")
            team_id = obj.get("team_id")
            if track_id is None or team_id is None:
                continue
            key = str(track_id)
            team_counters.setdefault(key, Counter())[team_id] += 1

    resolved_team = {
        track_id: counter.most_common(1)[0][0]
        for track_id, counter in team_counters.items()
    }
    return resolved_team

# ==========================================
# PASS 1: METADATA & EVIDENCE COLLECTION
# ==========================================
def run_pass_1_evidence_collection(video_path, metadata_path):
    print("\n🚀 STARTING PASS 1: Tracking, Team Clustering & Evidence Collection...")

    results = model.track(
        source=video_path,
        save=False,
        imgsz=[1920, 1088],
        conf=PLAYER_CONF_THRESH,
        iou=0.45,
        stream=True,
        verbose=False,
        tracker="botsort.yaml"
    )

    metadata = {"frames": {}, "track_votes": {}}


    team_anchors = None

    for frame_idx, result in enumerate(results):
        result = result.cpu()

        if result.boxes is None or len(result.boxes) == 0:
            metadata["frames"][str(frame_idx)] = []
            continue


        frame_data = []
        boxes = result.boxes
        orig_img = result.orig_img
        h_frame, w_frame = orig_img.shape[:2]

        players_crop = extract_player_crops(result, result.orig_img, player_class_id=2)

        # Run team assignment
        team_map = {}
        if players_crop:
            player_crops_list, team_anchors = assign_teams_by_anchor(players_crop, resize_dim=(64, 64), anchors= team_anchors)

            for p in player_crops_list:
                t_id = p.get('team_cluster', None)
                track_id = p['id']
                team_map[track_id] = t_id

                # Digits Extraction on Torso
                if t_id in [0, 1]:
                    crop_img = p['crop']
                    h_c, w_c, _ = crop_img.shape
                    back_region = crop_img[int(h_c * 0.15):int(h_c * 0.55), :]

                    if back_region.size > 0:
                        jn_res = jn_model.predict(
                            back_region,
                            imgsz=640,
                            conf=0.25,
                            verbose=False
                        )[0]

                        digit_str, confidence = extract_jersey_number_from_yolo(jn_res)


                        if digit_str and confidence >= SINGLE_FRAME_DIGIT_CONF:
                            str_tr_id = str(track_id)
                            if str_tr_id not in metadata["track_votes"]:
                                metadata["track_votes"][str_tr_id] = {}
                            curr_votes = metadata["track_votes"][str_tr_id]
                            curr_votes[digit_str] = curr_votes.get(digit_str, 0.0) + confidence


        # Record Frame Bounding Boxes Metadata
        for i in range(len(boxes)):
            cls_id = int(boxes.cls[i].item())
            x1, y1, x2, y2 = boxes.xyxy[i].numpy().astype(int)
            track_id = int(boxes.id[i].item()) if boxes.id is not None else None

            frame_data.append({
                "track_id": track_id,
                "cls_id": cls_id,
                "team_id": team_map.get(track_id, None),
                "bbox": [int(x1), int(y1), int(x2), int(y2)]
            })

        metadata["frames"][str(frame_idx)] = frame_data

        if frame_idx % 100 == 0:
            print(f"   Processed Frame {frame_idx}...")

    # Save collected metadata JSON file
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"✅ PASS 1 COMPLETE: Saved metadata to {metadata_path}")


# ==========================================
# PASS 2: GLOBAL IDENTITY RESOLUTION
# ==========================================
import json
from collections import Counter


def resolve_team_by_majority(metadata):
    team_counters = {}

    for frame_idx, objs in metadata["frames"].items():
        for obj in objs:
            track_id = obj.get("track_id")
            team_id = obj.get("team_id")
            if track_id is None or team_id is None:
                continue
            key = str(track_id)
            team_counters.setdefault(key, Counter())[team_id] += 1

    resolved_team = {
        track_id: counter.most_common(1)[0][0]
        for track_id, counter in team_counters.items()
    }
    return resolved_team


def run_pass_2_identity_resolution(metadata_path):
    print("\n🧠 STARTING PASS 2: Hindsight Team Correction & Jersey Resolution...")

    with open(metadata_path, 'r') as f:
        metadata = json.load(f)

    resolved_team = resolve_team_by_majority(metadata)

    track_votes = metadata.get("track_votes", {})
    track_best_jersey = {}
    for track_id_str, votes in track_votes.items():
        if not votes:
            continue
        best_jersey = max(votes, key=votes.get)
        total_weight = votes[best_jersey]
        track_best_jersey[track_id_str] = (best_jersey, total_weight)

    resolved_tracks = {}
    by_team = {"0": [], "1": []}

    for track_id_str, (best_jersey, total_weight) in track_best_jersey.items():
        team_id = resolved_team.get(track_id_str)
        if team_id is None:
            continue
        by_team[str(team_id)].append((track_id_str, best_jersey, total_weight))

    MIN_GLOBAL_ACCUMULATED_WEIGHT = 0.5

    for team_id_str, entries in by_team.items():

        entries.sort(key=lambda e: e[2], reverse=True)
        assigned_jerseys_on_team = {}

        for track_id_str_, best_jersey, total_weight in entries:
            key = f"{team_id_str}:{track_id_str_}"
            if total_weight < MIN_GLOBAL_ACCUMULATED_WEIGHT:
                resolved_tracks[key] = ""
                continue
            resolved_tracks[key] = best_jersey
            assigned_jerseys_on_team[best_jersey] = track_id_str_
            #if best_jersey not in assigned_jerseys_on_team:
            #    resolved_tracks[key] = best_jersey
            #    assigned_jerseys_on_team[best_jersey] = track_id_str_
            #else:
            #    resolved_tracks[key] = "Conflict"

    print(f"✅ PASS 2 COMPLETE: Resolved {len(resolved_tracks)} tracks "
          f"({len(resolved_team)} tracks got a hindsight team correction).")

    return metadata, resolved_tracks, resolved_team



# ==========================================
# PASS 3: OFFLINE RENDERING & VIDEO WRITER
# ==========================================
def run_pass_3_video_rendering(video_path, output_path, metadata, resolved_tracks, resolved_team):
    print("\n🎬 STARTING PASS 3: Final Frame Rendering & Encoding...")

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_data = metadata["frames"].get(str(frame_idx), [])

        for obj in frame_data:
            cls_id = obj["cls_id"]
            track_id = obj["track_id"]
            team_id = resolved_team.get(str(track_id), obj["team_id"])

            x1, y1, x2, y2 = obj["bbox"]

            if cls_id == 2:  # Player
                key = f"{team_id}:{track_id}"
                resolved_jersey = resolved_tracks.get(key, "")

                color = TEAM_COLORS.get(team_id, (255, 255, 255))
                label = f"Player {resolved_jersey} | Team {team_id if team_id is not None else 'N/A'}"
            else:
                color = CLASS_COLORS.get(cls_id, (255, 255, 255))
                label_names = {0: "Ball", 1: "Goalkeeper", 3: "Referee"}
                track_str = f" ID: {track_id}" if track_id is not None else ""
                label = f"{label_names.get(cls_id, 'Unknown')}{track_str}"

            # Render Bounding Box and Label
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label_size, base_line = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            y1_label = max(y1, label_size[1] + 10)

            cv2.rectangle(frame, (x1, y1_label - label_size[1] - 5), (x1 + label_size[0], y1_label + base_line), color, cv2.FILLED)
            cv2.putText(frame, label, (x1, y1_label - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f"   Rendered Frame {frame_idx}...")

    cap.release()
    writer.release()
    print(f"🏁 PASS 3 COMPLETE: Final Tracked Match Output Saved to {output_path}")




In [25]:

run_pass_1_evidence_collection(VIDEO_PATH, METADATA_SAVE_PATH)

metadata, resolved_tracks, resolved_team = run_pass_2_identity_resolution(METADATA_SAVE_PATH)

run_pass_3_video_rendering(VIDEO_PATH, OUTPUT_VIDEO_PATH, metadata, resolved_tracks, resolved_team)


🚀 STARTING PASS 1: Tracking, Team Clustering & Evidence Collection...
Automatically initializing team color anchors from the first frame...
   Processed Frame 0...
   Processed Frame 100...
   Processed Frame 200...
   Processed Frame 300...
   Processed Frame 400...
   Processed Frame 500...
   Processed Frame 600...
   Processed Frame 700...
✅ PASS 1 COMPLETE: Saved metadata to /content/video_tracking_metadata.json

🧠 STARTING PASS 2: Hindsight Team Correction & Jersey Resolution...
✅ PASS 2 COMPLETE: Resolved 62 tracks (151 tracks got a hindsight team correction).

🎬 STARTING PASS 3: Final Frame Rendering & Encoding...
   Rendered Frame 100...
   Rendered Frame 200...
   Rendered Frame 300...
   Rendered Frame 400...
   Rendered Frame 500...
   Rendered Frame 600...
   Rendered Frame 700...
🏁 PASS 3 COMPLETE: Final Tracked Match Output Saved to /content/output_tracked_match2.mp4
